# FLUKE Coreference Resolution with OpenAI o3-2025-04-16 Reasoning Model

This notebook evaluates coreference resolution robustness using OpenAI's o3-2025-04-16 reasoning model with FLUKE linguistic modifications.

In [1]:
from datasets import load_dataset
import dspy
import openai
import os
import re
import pandas as pd
import json
from dotenv import load_dotenv
import glob
from scipy import stats
import time
from tqdm import tqdm

/Users/hungthinh/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

## Model Configuration

Configure o3-2025-04-16 reasoning model with different reasoning strategies.

In [4]:
# Available o3 and o1 reasoning models
REASONING_MODELS = {
    'o3-2025-04-16': 'openai/o3-2025-04-16',
    'o1-preview': 'openai/o1-preview',
    'o1-mini': 'openai/o1-mini', 
    'o1': 'openai/o1',
}

# Model selection with different reasoning strategies
REASONING_CONFIGS = {
    'standard': {
        'model': 'o3-2025-04-16',
        'instruction_style': 'standard',
        'description': 'Standard reasoning approach with o3'
    },
    'detailed': {
        'model': 'o3-2025-04-16',
        'instruction_style': 'detailed',
        'description': 'Detailed step-by-step reasoning with o3'
    },
    'efficient': {
        'model': 'o1-mini',
        'instruction_style': 'concise',
        'description': 'Efficient reasoning with o1-mini'
    }
}

# Select configuration
CONFIG_NAME = 'standard'  # Change to 'detailed' or 'efficient'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]
INSTRUCTION_STYLE = config['instruction_style']

print(f"Using configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Instruction style: {INSTRUCTION_STYLE}")
print(f"Description: {config['description']}")

Using configuration: standard
Model: o3-2025-04-16 (openai/o3-2025-04-16)
Instruction style: standard
Description: Standard reasoning approach with o3


In [5]:
# Configure DSPy
lm = dspy.LM(MODEL_ID, temperature=1, max_tokens=5000)
dspy.configure(lm=lm)

## Load Coreference Data

In [6]:
# Load coreference dataset
ds = pd.read_json('../../../data/train_dev_test_data/coref/test.json')
ds = ds.to_dict('records')

print(f"Loaded {len(ds)} coreference samples")
print(f"Sample structure: {list(ds[0].keys())}")

Loaded 1517 coreference samples
Sample structure: ['label', 'candidates', 'pronoun', 'text']


In [7]:
def remove_space(text):
    """Clean up spacing and formatting in text."""
    # Remove multiple spaces
    text = ' '.join(text.split())
    
    # Fix spacing around punctuation
    text = re.sub(r'\s+([.,!?])', r'\1', text)
    text = re.sub(r'([.,!?])\s+', r'\1 ', text)
    
    # Fix contractions
    text = re.sub(r'\s*\'\s*s\b', "'s", text)
    text = re.sub(r'\s*n\s*\'\s*t\b', "n't", text)
    text = re.sub(r'\s*\'\s*ve\b', "'ve", text)
    text = re.sub(r'\s*\'\s*re\b', "'re", text)
    text = re.sub(r'\s*\'\s*ll\b', "'ll", text)
    text = re.sub(r'\s*\'\s*d\b', "'d", text)
    text = re.sub(r'\s*\'\s*m\b', "'m", text)
    
    # Fix spaces around parentheses
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    
    # Remove spaces before and after text
    text = text.strip()
    
    return text

In [8]:
# Format examples for DSPy
examples = [
    dspy.Example({ 
                  "text": remove_space(r["text"]), 
                  "pronoun": r['pronoun'],
                  "candidate": '0: ' + str(r['candidates'][0]) + ', 1: ' + str(r['candidates'][1]),
                  "label": r['label']
                }).with_inputs("text", 'pronoun', 'candidate') 
    for r in ds
]

In [9]:
example = examples[0]
for k, v in example.items():
    print(f"\n{k.upper()}:\n")
    print(v)


TEXT:

Sabina is trying to look for Maria, but to no avail, as she worries for her safety.

PRONOUN:

she

CANDIDATE:

0: Sabina, 1: Maria

LABEL:

0


In [10]:
def extract_prediction(text):
    """Extract prediction from o3 model output."""
    matches = re.findall(r'\b[0-2]\b', text)
    parsed_answer = matches[-1] if matches else ""
    return parsed_answer

In [11]:
def eval_metric(true, prediction, trace=None):
    """Evaluate coreference prediction."""
    pred = prediction.label
    matches = re.findall(r'\b[0-9]\b', pred)
    parsed_answer = matches[-1] if matches else ""
    return parsed_answer == str(true.label)

# Evaluate the original test set

In [12]:
from dspy.evaluate import Evaluate

## Coreference Resolution with o3 Model

In [13]:
class O3Coref(dspy.Signature):
    """Which candidate does the pronoun refer to? Think step by step about the context, grammatical relationships, and semantic meaning. Answer with either 0 or 1."""
    text = dspy.InputField()
    pronoun = dspy.InputField()
    candidate = dspy.InputField()
    label = dspy.OutputField(desc="The index 0 or 1 of the candidates.", prefix='Answer:')

In [14]:
class O3CorefModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Coref)

    def forward(self, text, pronoun, candidate):
        return self.prog(text=text, pronoun=pronoun, candidate=candidate)

In [15]:
o3_coref = O3CorefModule()

In [16]:
# Test with a single example
pred = o3_coref(text=example.text, pronoun=example.pronoun, candidate=example.candidate)
print("\nTEST EXAMPLE:\n")
print(f"Text: {example.text}")
print(f"Pronoun: {example.pronoun}")
print(f"Candidates: {example.candidate}")
print(f"True Label: {example.label}")
print("\nPREDICTION:\n")
print(pred)
print(f"\nCorrect: {eval_metric(example, pred)}")


TEST EXAMPLE:

Text: Sabina is trying to look for Maria, but to no avail, as she worries for her safety.
Pronoun: she
Candidates: 0: Sabina, 1: Maria
True Label: 0

PREDICTION:

Prediction(
    label='0'
)

Correct: True


## Evaluate Original Test Set

In [17]:
# Use subset for testing due to o3 costs and rate limits
test_examples = examples[:100]  # Adjust size as needed

print(f"Evaluating on {len(test_examples)} examples")

evaluate = Evaluate(
    devset=test_examples, 
    metric=eval_metric, 
    num_threads=1,  # Lower for o3 models
    display_progress=True, 
    display_table=10, 
    return_outputs=True, 
    return_all_scores=True
)

results = evaluate(o3_coref)

# Save results
items = []
for sample in results[1]:
    item = {
        'text': sample[0]['text'],
        'pronoun': sample[0]['pronoun'],
        'candidate': sample[0]['candidate'],
        'label': sample[0]['label'],
        'pred': extract_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']  # Save full reasoning
    }
    items.append(item)

df_result = pd.DataFrame(data=items)
df_result.to_csv(f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv', index=False)
print(f"Results saved to results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv")
print(f"Accuracy: {results[0]:.3f}")

Evaluating on 100 examples
Average Metric: 78.00 / 100 (78.0%): 100%|██████████| 100/100 [08:24<00:00,  5.05s/it]

2025/08/11 15:07:12 INFO dspy.evaluate.evaluate: Average Metric: 78 / 100 (78.0%)


,text,pronoun,candidate,example_label,pred_label,eval_metric
0,"Sabina is trying to look for Maria, but to no avail, as she worrie...",she,"0: Sabina, 1: Maria",0,0,✔️ [True]
1,I'm sure that my map will show this building; it is very famous.,it,"0: The map, 1: The building",1,1,✔️ [True]
2,"Though Gino tries to avoid Ross, he can not help but eventually fa...",he,"0: Gino, 1: Ross",0,0,✔️ [True]
3,De Vries kills Yueh but he also dies with Leto in the assassinatio...,he,"0: Yueh, 1: De Vries",1,1,✔️ [True]
4,"Hair Stylists transformed the Cowboy's Cheerleaders into beauties,...",they,"0: Hair Stylists, 1: the Cowboy's Cheerleaders",0,0,✔️ [True]
5,Ed Helms was cast as Derek Smeathe but scheduling conflicts preven...,him,"0: Derek Smeathe, 1: Ed Helms",1,1,✔️ [True]
6,"Dante eventually proposed to Frank, although he was still in love ...",he,"0: Dante, 1: Frank",0,0,✔️ [True]
7,Thomson visited Cooper's grave in 1765. At that date he had been t...,he,"0: Thomson, 1: Cooper",0,0,✔️ [True]
8,Some enthusiastic Black Friday shoppers are already lining up at r...,they,"0: Some enthusiastic Black Friday shoppers, 1: retail stores",0,0,✔️ [True]
9,Jane gave Joan candy because she wasn't hungry.,she,"0: Jane, 1: Joan",0,0,✔️ [True]


OSError: Cannot save file into a non-existent directory: 'results/coref'

## Chain-of-Thought with o3 (Enhanced Reasoning)

In [ ]:
class CoTO3Coref(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(O3Coref)

    def forward(self, text, pronoun, candidate):
        return self.prog(text=text, pronoun=pronoun, candidate=candidate)

In [ ]:
cot_o3_coref = CoTO3Coref()
pred_cot = cot_o3_coref(text=example.text, pronoun=example.pronoun, candidate=example.candidate)
print("\nCHAIN-OF-THOUGHT EXAMPLE:\n")
print(f"Text: {example.text}")
print(f"Pronoun: {example.pronoun}")
print(f"Candidates: {example.candidate}")
print("\nCOT PREDICTION:\n")
print(pred_cot)

In [ ]:
# Evaluate CoT version (optional)
evaluate_cot = Evaluate(
    devset=test_examples[:50], 
    metric=eval_metric, 
    num_threads=1, 
    display_progress=True, 
    display_table=5, 
    return_outputs=True, 
    return_all_scores=True
)

results_cot = evaluate_cot(cot_o3_coref)

items_cot = []
for sample in results_cot[1]:
    item = {
        'text': sample[0]['text'],
        'pronoun': sample[0]['pronoun'],
        'candidate': sample[0]['candidate'],
        'rationale': sample[1].get('reasoning', ''),
        'label': sample[0]['label'],
        'pred': extract_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']
    }
    items_cot.append(item)

df_result_cot = pd.DataFrame(data=items_cot)
df_result_cot.to_csv(f'{MODEL_NAME}-{CONFIG_NAME}-0shot-cot-coref.csv', index=False)
print(f"CoT Accuracy: {results_cot[0]:.3f}")

# Evaluate by modification

## Without label change

In [18]:
def evaluate_modified_set(ds, program, max_samples=50):
    """Evaluate on modified dataset with sample limit."""
    # Limit samples due to o3 cost and rate limits
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "text": remove_space(r['modified_text']), 
                      "original_text": remove_space(r['original_text']),
                      "pronoun": r['modified_pronoun'],
                      "candidate": '0: ' + str(r['modified_candidates'][0]) + ', 1: ' + str(r['modified_candidates'][1]),
                      "label": int(r['modified_label']),
                      "modified_label": int(r['modified_label'])
                    }).with_inputs("text", "pronoun", "candidate") 
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True
    )
    
    return evaluate(program)

In [19]:
# Recreate classes for consistency
class O3Coref(dspy.Signature):
    """Which candidate does the pronoun refer to? Answer with either 0 or 1."""
    text = dspy.InputField()
    pronoun = dspy.InputField()
    candidate = dspy.InputField()
    label = dspy.OutputField(desc="The index 0 or 1 of the candidates.", prefix='Answer:')

class O3CorefModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Coref)

    def forward(self, text, pronoun, candidate):
        return self.prog(text=text, pronoun=pronoun, candidate=candidate)
        
o3_coref = O3CorefModule()

In [21]:
# Configure o3 model and load original predictions
lm = dspy.LM(MODEL_ID, temperature=1, max_tokens=5000)
dspy.configure(lm=lm)

# Load original predictions for comparison
original_pred_file = f'../results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file, index_col=False)
    original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
    print(f"Loaded original predictions from {original_pred_file}")
else:
    print(f"Original predictions file not found: {original_pred_file}")
    print("Please run the original evaluation first")
    original_pred_ds = None

# Get modification files (subset for testing)
json_files = glob.glob('../data/modified_data/coref/*_100.json')
test_modifications = ['typo_bias_100.json', 'capitalization_100.json', 'punctuation_100.json']
json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"Testing modifications: {[f.split('/')[-1] for f in json_files]}")

for json_file in json_files:
    print(f"\nProcessing: {json_file}")
    if 'grammatical_role' in json_file or 'negation' in json_file:
        print("Skipping complex modification for now")
        continue
        
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    results_modified = evaluate_modified_set(data, o3_coref, max_samples=20)
    
    # Convert results to dataframe
    items = []
    for sample in results_modified[1]:
        item = {}
        modified_text = sample[0]['text']
        original_text = sample[0]['original_text']
        pred = sample[1]['label']
        
        original_text = remove_space(original_text)
        pred = extract_prediction(pred)
        
        # Find original prediction
        original_pred = None
        if original_pred_ds is not None:
            matching_rows = original_pred_ds[original_pred_ds['text'] == original_text]
            if not matching_rows.empty:
                original_pred = matching_rows.iloc[0]['pred']
        
        item['original_text'] = original_text
        item['modified_text'] = modified_text
        item['modified_pronoun'] = sample[0]['pronoun']
        item['modified_candidates'] = sample[0]['candidate']
        item['modified_label'] = sample[0]['modified_label']
        item['modified_pred'] = pred
        item['original_pred'] = original_pred
        item['original_label'] = sample[0]['label']
        item['raw_output'] = sample[1]['label']
        items.append(item)
    
    df_result = pd.DataFrame(data=items)
    
    # Save results
    output_filename = f"results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-{json_file.split('/')[-1].replace('.json', '')}.csv"
    df_result.to_csv(output_filename, index=False)
    print(f"Saved results to: {output_filename}")
    print(f"Accuracy: {results_modified[0]:.3f}")
    
    # Add delay to respect rate limits
    time.sleep(5)

Original predictions file not found: ../results/coref/o3-2025-04-16-standard-0shot-coref.csv
Please run the original evaluation first
Testing modifications: []


## With label change

In [ ]:
def evaluate_modified_set_with_label_change(ds, program, max_samples=50):
    """Evaluate on modified dataset where labels might change."""
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "text": remove_space(r['modified_text']), 
                      "original_text": remove_space(r['original_text']),
                      "pronoun": r['modified_pronoun'],
                      "candidate": "0: " + str(r['modified_candidates'][0]) + ", 1: " + str(r['modified_candidates'][1]),
                      "label": int(r['modified_label']),
                      "original_label": int(r['original_label']),
                      "original_pronoun": r['original_pronoun'],
                      "type": r['type'],
                      "index": r.get('index', 0)
                    }).with_inputs("text", "pronoun", "candidate") 
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Test modifications that change labels
json_files_label_change = glob.glob('../data/modified_data/coref/*_100.json')
label_change_modifications = ['active_to_passive_100.json']
json_files_label_change = [f for f in json_files_label_change if any(mod in f for mod in label_change_modifications)]

for json_file in json_files_label_change:
    print(f"\nProcessing label-changing modification: {json_file}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    results_modified = evaluate_modified_set_with_label_change(data, o3_coref, max_samples=15)
    
    # Convert results to dataframe
    items = []
    for sample in results_modified[1]:
        item = {}
        modified_text = sample[0]['text']
        original_text = sample[0]['original_text']
        label = sample[0]['label']
        pred = sample[1]['label'] if sample[1].get('label') is not None else "[]"
        
        original_text = remove_space(original_text)
        pred = extract_prediction(pred)
        
        # Find original prediction by index if available
        original_pred = None
        if original_pred_ds is not None:
            try:
                if 'index' in sample[0] and sample[0]['index'] < len(original_pred_ds):
                    original_pred = original_pred_ds.iloc[sample[0]['index']]['pred']
                else:
                    # Fallback to text matching
                    matching_rows = original_pred_ds[original_pred_ds['text'] == original_text]
                    if not matching_rows.empty:
                        original_pred = matching_rows.iloc[0]['pred']
            except:
                original_pred = None
        
        item['original_text'] = original_text
        item['modified_text'] = modified_text
        item['modified_label'] = sample[0]['label']
        item['modified_pred'] = pred
        item['original_pred'] = original_pred
        item['modified_pronoun'] = sample[0]['pronoun']
        item['modified_candidates'] = sample[0]['candidate']
        item['original_label'] = sample[0]['original_label']
        item['type'] = sample[0]['type']
        item['raw_output'] = sample[1]['label']
        items.append(item)
    
    df_result = pd.DataFrame(data=items)
    
    # Save results
    output_filename = f"results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-{json_file.split('/')[-1].replace('.json', '')}.csv"
    df_result.to_csv(output_filename, index=False)
    print(f"Saved results to: {output_filename}")
    print(f"Accuracy: {results_modified[0]:.3f}")
    
    time.sleep(5)

# Aggregate results

In [ ]:
from scipy import stats

In [ ]:
# Aggregate results across all modifications
result_files = glob.glob(f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')
aggregated_results = []

print(f"Found {len(result_files)} result files for {MODEL_NAME}-{CONFIG_NAME}")

for file in result_files:
    # Extract modification type from filename
    mod_type = file.split('-')[-1].replace('.csv', '')
    
    try:
        # Read results file
        df = pd.read_csv(file)
        
        # Handle missing columns gracefully
        if 'original_pred' not in df.columns or df['original_pred'].isna().all():
            print(f"Warning: {file} missing original predictions")
            continue
            
        # Calculate accuracies
        original_correct = (df['original_pred'] == df['original_label']).sum()
        modified_correct = (df['modified_pred'] == df['modified_label']).sum()
        total = len(df)

        if total == 0:
            continue
            
        original_acc = original_correct / total
        modified_acc = modified_correct / total
        
        # Calculate the difference
        difference = -round(original_acc - modified_acc, 2)
        
        # Calculate percentage difference
        if original_correct > 0:
            pct_difference = -round((original_correct - modified_correct) / original_correct * 100, 2)
        else:
            pct_difference = 0
        
        # Perform t-test if we have enough data
        try:
            t_stat, p_value = stats.ttest_ind(
                (df['original_pred'] == df['original_label']).astype(float),
                (df['modified_pred'] == df['modified_label']).astype(float)
            )
        except:
            p_value = None
        
        aggregated_results.append({
            'task': 'coreference_resolution',
            'model': f'{MODEL_NAME}-{CONFIG_NAME}',
            'modification': mod_type,
            'original_res': round(original_acc, 3),
            'modified_res': round(modified_acc, 3),
            'difference': difference,
            'pct_difference': pct_difference,
            'p_value': p_value,
            'samples': total
        })
        
    except Exception as e:
        print(f"Error processing {file}: {e}")
        continue

# Create final results dataframe
if aggregated_results:
    results_df = pd.DataFrame(aggregated_results)
    
    # Sort the results
    modification_order = ['temporal_bias_100', 'geographical_bias_100', 'length_bias_100', 
                         'typo_bias_100', 'capitalization_100', 'punctuation_100', 
                         'derivation_100', 'compound_word_100', 'active_to_passive_100',
                         'grammatical_role_100', 'coordinating_conjunction_100', 
                         'concept_replacement_100', 'negation_100', 'discourse_100',
                         'sentiment_100', 'casual_100', 'dialectal_100']
    
    # Only use modifications that exist in our results
    existing_mods = results_df['modification'].unique()
    modification_order = [mod for mod in modification_order if mod in existing_mods]
    
    results_df['modification'] = pd.Categorical(results_df['modification'], categories=modification_order, ordered=True)
    results_df = results_df.sort_values(by='modification')

    # Calculate averages across all modifications
    avg_original = results_df['original_res'].mean()
    avg_modified = results_df['modified_res'].mean()
    avg_difference = avg_original - avg_modified
    avg_pct_difference = results_df['pct_difference'].mean()

    # Add averages as a new row
    avg_row = {
        'task': 'coreference_resolution',
        'model': f'{MODEL_NAME}-{CONFIG_NAME}',
        'modification': 'average',
        'original_res': round(avg_original, 3),
        'modified_res': round(avg_modified, 3),
        'difference': -round(avg_difference, 3),
        'pct_difference': round(avg_pct_difference, 2),
        'p_value': None,
        'samples': results_df['samples'].sum()
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([avg_row])], ignore_index=True)

    print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
    print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])

    # Save aggregated results
    results_df.to_csv(f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-DP.csv', index=False)
    print(f"\nAggregated results saved to: results/coref/{MODEL_NAME}-{CONFIG_NAME}-DP.csv")

    # Apply styling to highlight performance drops
    def highlight_drops_and_significance(row):
        colors = [''] * len(row)
        if row['original_res'] > row['modified_res']:
            colors = ['background-color: red'] * len(row)
            # If p-value < 0.05, add bold text
            if 'p_value' in row and row['p_value'] is not None and row['p_value'] < 0.05:
                colors = ['background-color: red; font-weight: bold'] * len(row)
        return colors

    styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
    display(styled_df)
    
else:
    print("No results found to aggregate")

## Model Comparison and Analysis

In [ ]:
# Compare with other models if available
comparison_files = {
    'Llama-405B': 'results/coref/llama-0shot-coref.csv',
    'Claude-3.5': 'results/coref/claude-3-5-sonnet-0shot-coref.csv',
    'Mixtral-8x22B': 'results/coref/mixtral-8x22b-0shot-coref.csv',
    f'{MODEL_NAME}-{CONFIG_NAME}': f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv'
}

model_accuracies = {}
for model_name, file_path in comparison_files.items():
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            # Handle different column names
            pred_col = 'pred' if 'pred' in df.columns else 'prediction'
            if pred_col in df.columns and 'label' in df.columns:
                accuracy = (df[pred_col] == df['label']).mean()
                model_accuracies[model_name] = accuracy
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

# Display comparison
if model_accuracies:
    comparison_df = pd.DataFrame([
        {'Model': model, 'Accuracy': acc, 'Performance': f"{acc:.1%}"} 
        for model, acc in model_accuracies.items()
    ])
    comparison_df = comparison_df.sort_values('Accuracy', ascending=False)
    
    print("\nModel Comparison on Coreference Resolution:")
    print(comparison_df)

    # Highlight o3 performance
    o3_model_key = f'{MODEL_NAME}-{CONFIG_NAME}'
    o3_performance = model_accuracies.get(o3_model_key, 0)
    print(f"\n{o3_model_key} Accuracy: {o3_performance:.3f} ({o3_performance:.1%})")
    
    if len(model_accuracies) > 1:
        other_models = [acc for model, acc in model_accuracies.items() if model != o3_model_key]
        if other_models:
            avg_others = sum(other_models) / len(other_models)
            improvement = o3_performance - avg_others
            print(f"Average of other models: {avg_others:.3f} ({avg_others:.1%})")
            print(f"Performance difference: {improvement:+.3f} ({improvement:+.1%})")

    # Style the dataframe
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]

    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No model comparison data available")

## o3 Reasoning Analysis

In [ ]:
# Analyze o3 reasoning quality (if raw outputs are available)
if 'raw_output' in df_result.columns and not df_result.empty:
    print("Sample o3 Reasoning for Coreference Resolution:")
    print("=" * 60)
    
    for i, (idx, row) in enumerate(df_result.head(3).iterrows()):
        print(f"\nExample {i+1}:")
        print(f"Text: {row['text'][:150]}{'...' if len(row['text']) > 150 else ''}")
        print(f"Pronoun: {row.get('pronoun', row.get('modified_pronoun', 'N/A'))}")
        print(f"Candidates: {row.get('candidate', row.get('modified_candidates', 'N/A'))}")
        print(f"True Label: {row.get('label', row.get('modified_label', 'N/A'))}")
        print(f"Prediction: {row.get('pred', row.get('modified_pred', 'N/A'))}")
        print(f"Reasoning: {row['raw_output'][:600]}{'...' if len(str(row['raw_output'])) > 600 else ''}")
        print("-" * 50)

# Summary statistics
print(f"\n{MODEL_NAME}-{CONFIG_NAME} Evaluation Summary:")
print("=" * 60)

if 'results' in locals():
    print(f"Base accuracy on coreference resolution: {results[0]:.3f} ({results[0]:.1%})")

if 'aggregated_results' in locals() and aggregated_results:
    avg_robustness = sum([r['difference'] for r in aggregated_results if r['difference'] is not None]) / len([r for r in aggregated_results if r['difference'] is not None])
    print(f"Average robustness impact: {avg_robustness:+.3f}")
    print(f"Modifications tested: {len(aggregated_results)}")

print(f"\nKey insights with {MODEL_NAME}:")
print(f"- Enhanced reasoning model with sophisticated coreference analysis")
print(f"- Detailed reasoning traces show pronoun-antecedent relationship analysis")
print(f"- Performance on linguistic robustness varies by modification complexity")
print(f"- Advanced reasoning helps with ambiguous cases and context understanding")

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Coreference Resolution Evaluation with {MODEL_NAME} Complete!")
print(f"{'='*60}")
print(f"Configuration: {config['description']}")
print(f"Files saved in results/coref/ with prefix '{MODEL_NAME}-{CONFIG_NAME}-'")
print(f"\nNext steps:")
print(f"1. Review reasoning traces for coreference resolution strategies")
print(f"2. Compare robustness with other models on different modifications")
print(f"3. Analyze which linguistic changes most challenge {MODEL_NAME}")
print(f"4. Consider different reasoning configurations (detailed vs standard)")
print(f"5. Evaluate cost-benefit for coreference resolution tasks")